In [ ]:
# ====================================================
# SYNTHÈSE VOCALE EN MALGACHE - MODÈLE MMS (Meta)
# Copiez et collez ce bloc dans une cellule Colab
# ====================================================

# 1. Installation des bibliothèques nécessaires
!pip install -q transformers scipy #accelerate

# 2. Imports
import torch
import scipy.io.wavfile as wav
from IPython.display import Audio
from transformers import VitsModel, AutoTokenizer

# 3. Chargement du modèle (spécifique au malgache)
print("Chargement du modèle MMS TTS pour le malgache...")
model = VitsModel.from_pretrained("facebook/mms-tts-mlg")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-mlg")
print("✅ Modèle chargé avec succès.")

# 4. Saisie du texte à synthétiser (modifiez ici votre texte)
texte_malgache = "very  pory zahay eh"  # Exemple : Bonjour, comment allez-vous ?

# 5. Tokenisation et génération de l'audio
inputs = tokenizer(texte_malgache, return_tensors="pt")
with torch.no_grad():
    output = model(**inputs).waveform

# 6. Lecture audio dans le notebook
audio_numpy = output.squeeze().numpy()
frequence = model.config.sampling_rate
print(f"Génération terminée. Lecture à {frequence} Hz...")
display(Audio(audio_numpy, rate=frequence))

# 7. Sauvegarde dans un fichier (optionnel)
fichier_sortie = "parole_malgache.wav"
wav.write(fichier_sortie, rate=frequence, data=audio_numpy)
print(f"✅ Audio sauvegardé sous '{fichier_sortie}'")

Chargement du modèle MMS TTS pour le malgache...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/329 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

✅ Modèle chargé avec succès.
Génération terminée. Lecture à 16000 Hz...


✅ Audio sauvegardé sous 'parole_malgache.wav'


In [ ]:
# import joblib

# with open("mms_model.pkl", "wb") as model_file:
#     joblib.dump(model, model_file)

# with open("mms_tokenizer.pkl", "wb") as tokenizer_file:
#     joblib.dump(AutoTokenizer, tokenizer_file)
# torch.save(model.state_dict(),"./mms_model.pkl")
# torch.save(tokenizer, "./mms_tokenizer.pkl")
model.save_pretrained("./mms_model")
tokenizer.save_pretrained("./mms_tokenizer")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./mms_tokenizer/tokenizer_config.json',
 './mms_tokenizer/vocab.json',
 './mms_tokenizer/added_tokens.json')

### Note
d'apres Grok, le moyen le plus fiable pour telecharger un modele est
- model.save_pretrained("path/to/saving/location")
- installer transformers via pip en local pour appeller les classes des modeles(VitsModel,Autotokenizer),
- d'appeller ces modeles en local et de les passer les modeles preentrainees downloaded avec ModelClass.from_pretrained("local/path/to/folder")
- Then locally load with from_pretrained("./my_inference_model", local_files_only=True)


In [ ]:
!zip -r mms_model.zip mms_model
!zip -r mms_tokenizer.zip mms_tokenizer

  adding: mms_model/ (stored 0%)
  adding: mms_model/config.json (deflated 65%)
  adding: mms_model/model.safetensors (deflated 7%)
  adding: mms_tokenizer/ (stored 0%)
  adding: mms_tokenizer/added_tokens.json (stored 0%)
  adding: mms_tokenizer/tokenizer_config.json (deflated 61%)
  adding: mms_tokenizer/vocab.json (deflated 53%)


----------

In [ ]:
# ====================================================
# CORRECTION : Extraction des versets depuis le XML
# ====================================================

import requests
import xml.etree.ElementTree as ET
import re
import csv
import os
from tqdm import tqdm

# 1. Téléchargement (déjà fait, mais on recommence proprement)
url = "https://github.com/Beblia/Holy-Bible-XML-Format/raw/refs/heads/master/MalagasyMRVBible.xml"
response = requests.get(url)
with open("malagasy_bible.xml", "wb") as f:
    f.write(response.content)

# 2. Fonction pour explorer la structure XML
def explore_xml(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()

    # Afficher la balise racine et les premiers enfants
    print(f"Balise racine : {root.tag}")
    print(f"Namespace : {root.tag.split('}')[0].strip('{') if '}' in root.tag else 'Aucun'}")

    # Lister les balises des premiers éléments
    tags = set()
    for elem in root.iter():
        tag = elem.tag.split('}')[-1]  # enlever namespace
        tags.add(tag)
    print(f"Balises trouvées : {sorted(tags)[:20]}")  # les 20 premières

    # Chercher toutes les balises qui contiennent du texte
    text_tags = []
    for elem in root.iter():
        if elem.text and elem.text.strip():
            tag = elem.tag.split('}')[-1]
            text_tags.append(tag)
    print(f"Balises avec texte : {set(text_tags)}")

    return root

# 3. Explorer le fichier pour comprendre sa structure
root = explore_xml("malagasy_bible.xml")

# 4. Extraction adaptative : on cherche toutes les balises qui ressemblent à des versets
#    On va chercher les balises 'v' (avec ou sans namespace) et aussi 'verse'
verses = []
for elem in root.iter():
    tag = elem.tag.split('}')[-1]  # nom local sans namespace
    if tag.lower() in ['v', 'verse', 'verset']:
        if elem.text and elem.text.strip():
            verses.append(elem.text.strip())

print(f"Nombre de versets extraits : {len(verses)}")

# Si toujours 0, on prend TOUS les éléments textuels (fallback)
if len(verses) == 0:
    print("Aucune balise 'v' trouvée. Récupération de tout le texte...")
    for elem in root.iter():
        if elem.text and elem.text.strip():
            # Éviter les répétitions et les longs textes (chapitres entiers)
            if len(elem.text.strip()) < 500:
                verses.append(elem.text.strip())
    print(f"Texte extrait de {len(verses)} éléments")

# 5. Nettoyage et normalisation
def normalize_text(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^\w\s\']', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

cleaned_verses = []
for v in tqdm(verses, desc="Nettoyage"):
    cleaned = normalize_text(v)
    if len(cleaned) > 10:
        cleaned_verses.append(cleaned)

print(f"Versets nettoyés : {len(cleaned_verses)}")

# 6. Segmentation en phrases
def split_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if len(s.strip()) > 5]

all_sentences = []
for v in tqdm(cleaned_verses, desc="Segmentation"):
    all_sentences.extend(split_sentences(v))

print(f"Phrases brutes : {len(all_sentences)}")

# 7. Filtrage
filtered = [s for s in all_sentences if 10 <= len(s) <= 200]
print(f"Phrases après filtrage (10-200 caractères) : {len(filtered)}")

# 8. Création du CSV
AUDIO_DIR = "audio_segments"
os.makedirs(AUDIO_DIR, exist_ok=True)

with open("metadata.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["file_name", "text"])
    for idx, sent in enumerate(filtered):
        writer.writerow([f"segment_{idx:05d}.wav", sent])

print("✅ metadata.csv généré avec", len(filtered), "entrées.")
print("\nExemples :")
for i in range(min(5, len(filtered))):
    print(f"{i+1}. {filtered[i]}")

Balise racine : bible
Namespace : Aucun
Balises trouvées : ['bible', 'book', 'chapter', 'testament', 'verse']
Balises avec texte : {'verse'}
Nombre de versets extraits : 31079


Nettoyage: 100%|██████████| 31079/31079 [00:00<00:00, 40065.45it/s]


Versets nettoyés : 31077


Segmentation: 100%|██████████| 31077/31077 [00:00<00:00, 76925.35it/s]


Phrases brutes : 31077
Phrases après filtrage (10-200 caractères) : 26294
✅ metadata.csv généré avec 26294 entrées.

Exemples :
1. tamin ny voalohany andriamanitra nahary ny lanitra sy ny tany
2. ary ny tany dia tsy nisy endrika sady foana ary haizina no teo ambonin ny lalina ary ny fanahin andriamanitra nanomba teo ambonin ny rano
3. ary andriamanitra niteny hoe misia mazava dia nisy mazava
4. ary hitan andriamanitra fa tsara ny mazava ka nosarahin andriamanitra ny mazava sy ny maizina
5. ary nataon andriamanitra hoe andro ny mazava ary ny maizina dia nataony hoe alina dia nisy hariva ary nisy maraina izay no andro voalohany
